# YZM212 Makine Öğrenmesi - 2. Laboratuvar Ödevi
## MLE ile Akıllı Şehir Planlaması

**Ad Soyad:** Hüseyin Baturalp Yiğit  
**Öğrenci No:** 23291030  
**Tarih:** 2026-03-15

---
## Bölüm 1: Teorik Türetme (Analitik Çözüm)

Poisson dağılımına göre bir dakikadaki araç sayısı $K$ rassal değişkeni için:

$$
P(K=k\mid\lambda)=\frac{e^{-\lambda}\lambda^k}{k!},\quad k=0,1,2,\ldots
$$

$n$ adet bağımsız gözlem $\{k_1,k_2,\ldots,k_n\}$ verildiğinde olabilirlik fonksiyonu:

$$
L(\lambda)=\prod_{i=1}^{n} \frac{e^{-\lambda}\lambda^{k_i}}{k_i!}
= e^{-n\lambda}\,\lambda^{\sum_{i=1}^{n}k_i}\,\prod_{i=1}^{n}\frac{1}{k_i!}
$$

Doğal logaritma alınırsa log-olabilirlik:

$$
\ell(\lambda)=\log L(\lambda)
= -n\lambda + \left(\sum_{i=1}^{n}k_i\right)\log\lambda - \sum_{i=1}^{n}\log(k_i!)
$$

MLE için türev sıfıra eşitlenir:

$$
\frac{d\ell(\lambda)}{d\lambda}
= -n + \frac{\sum_{i=1}^{n}k_i}{\lambda}=0
\Rightarrow
\hat{\lambda}_{MLE}=\frac{1}{n}\sum_{i=1}^{n}k_i=\bar{k}
$$

Sonuç olarak Poisson modeli altında en yüksek olabilirlik veren parametre, gözlemlerin aritmetik ortalamasıdır.

---
## Bölüm 2: Python ile Sayısal MLE

In [ ]:
import numpy as np
import scipy.optimize as opt
import scipy.stats as stats
import scipy.special as special
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11
})

print('Kütüphaneler yüklendi.')

In [ ]:
# 1 dakikada geçen araç sayısı gözlemleri
traffic_data = np.array([12, 15, 10, 8, 14, 11, 13, 16, 9, 12, 11, 14, 10, 15])
print(f'Veri boyutu: {len(traffic_data)}')
print(f'Ortalama (analitik MLE): {np.mean(traffic_data):.6f}')

In [ ]:
def poisson_nll(lam_arr, data):
    # scipy.optimize parametreyi array olarak yollar
    lam = lam_arr[0]

    if lam <= 0:
        return np.inf

    n = len(data)
    log_likelihood = (
        -n * lam
        + np.sum(data) * np.log(lam)
        - np.sum(special.gammaln(data + 1))
    )
    return -log_likelihood

In [ ]:
initial_guess = [1.0]
result = opt.minimize(
    poisson_nll,
    initial_guess,
    args=(traffic_data,),
    bounds=[(1e-6, None)]
)

lambda_num = result.x[0]
lambda_ana = np.mean(traffic_data)

print(f'Sayısal MLE : {lambda_num:.8f}')
print(f'Analitik MLE: {lambda_ana:.8f}')
print(f'Fark        : {abs(lambda_num - lambda_ana):.10f}')
print(f'Başarılı mı?: {result.success}')

### Kısa Yorum

Sayısal optimizasyonla bulunan $\hat{\lambda}$ değeri, analitik yöntemle bulunan ortalama değerle neredeyse aynıdır.
Bu beklenen bir durumdur; çünkü Poisson için MLE kapalı formda doğrudan ortalamaya eşit olur.
Küçük fark, optimize edicinin sayısal toleransları ve yuvarlama etkilerinden kaynaklanır.

---
## Bölüm 3: Model Karşılaştırma ve Görselleştirme

In [ ]:
lambda_hat = lambda_num
k_values = np.arange(0, int(np.max(traffic_data)) + 8)
pmf_values = stats.poisson.pmf(k_values, mu=lambda_hat)

unique_vals, counts = np.unique(traffic_data, return_counts=True)
relative_freq = counts / len(traffic_data)

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(unique_vals - 0.15, relative_freq, width=0.3, alpha=0.75,
       color='#2A9D8F', label='Gerçek Veri (Göreli Frekans)')
ax.bar(k_values + 0.15, pmf_values, width=0.3, alpha=0.75,
       color='#E76F51', label=f'Poisson PMF (lambda={lambda_hat:.2f})')

ax.set_xlabel('k (1 dakikadaki araç sayısı)')
ax.set_ylabel('Olasılık / Göreli Frekans')
ax.set_title('Poisson Modeli ve Trafik Verisi Karşılaştırması')
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig('figures/bolum3_model_karsilastirma.png', dpi=160, bbox_inches='tight')
plt.show()

### Görsel Yorum

Histogramdaki yoğunluk ile Poisson PMF değerlerinin zirve yaptığı bölge büyük ölçüde örtüşmektedir.
Özellikle orta aralıktaki gözlemler (yaklaşık 8-16 bandı) model tarafından tutarlı biçimde temsil edilmektedir.
Veri sayısı sınırlı olduğu için her noktada birebir eşleşme beklenmez; yine de genel şekil benzerliği modelin makul olduğunu destekler.

---
## Bölüm 4: Outlier Analizi

In [ ]:
outlier_value = 200
traffic_with_outlier = np.append(traffic_data, outlier_value)

result_outlier = opt.minimize(
    poisson_nll,
    [1.0],
    args=(traffic_with_outlier,),
    bounds=[(1e-6, None)]
)

lambda_normal = lambda_num
lambda_outlier = result_outlier.x[0]
pct_change = ((lambda_outlier / lambda_normal) - 1) * 100

print(f'Normal lambda : {lambda_normal:.6f}')
print(f'Outlier lambda: {lambda_outlier:.6f}')
print(f'Artış oranı   : %{pct_change:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

k_zoom = np.arange(0, 31)
axes[0].plot(k_zoom, stats.poisson.pmf(k_zoom, mu=lambda_normal),
             marker='o', color='#1D3557', label=f'Outlier yok (lambda={lambda_normal:.2f})')
axes[0].plot(k_zoom, stats.poisson.pmf(k_zoom, mu=lambda_outlier),
             marker='s', linestyle='--', color='#E63946', label=f'Outlier var (lambda={lambda_outlier:.2f})')
axes[0].set_xlabel('k')
axes[0].set_ylabel('PMF')
axes[0].set_title('PMF Kayması (0-30 Aralığı)')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].hist(traffic_data, bins=range(0, 25), density=True, alpha=0.6,
             color='#457B9D', label='Orijinal Veri')
axes[1].hist(traffic_with_outlier, bins=range(0, 210), density=True, alpha=0.45,
             color='#F4A261', label='Outlier Eklenmiş Veri')
axes[1].axvline(lambda_normal, color='#1D3557', linewidth=2, label='lambda normal')
axes[1].axvline(lambda_outlier, color='#E63946', linewidth=2, linestyle='--', label='lambda outlier')
axes[1].set_xlabel('Araç sayısı / dakika')
axes[1].set_ylabel('Yoğunluk')
axes[1].set_title('Outlier Etkisi ve MLE Kayması')
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.savefig('figures/bolum4_outlier_analizi.png', dpi=160, bbox_inches='tight')
plt.show()

### Outlier Tartışması

- MLE neden bu kadar hassas?: Poisson için $\hat{\lambda}$ doğrudan ortalama olduğu için tek bir aşırı büyük değer ortalamayı yukarı çeker.
- Belediye planlama hatası nasıl doğabilir?: Trafik gerçekte normal seviyedeyken model, yapay olarak yüksek talep varmış gibi görünebilir; bu da gereksiz kapasite artışı kararlarına yol açabilir.
- Bu riski azaltmak için ne yapılabilir?: Veri giriş doğrulama kuralları, aykırı değer filtreleme, medyan/trimmed mean gibi robust özetler ve daha uzun dönemli veriyle modelleme birlikte kullanılabilir.

---
## Kaynakça
- Bishop, C. M. Pattern Recognition and Machine Learning.
- SciPy ve NumPy dokümantasyonları.